<div style="border:3px solid black;
            border-radius:10px;
            background-color:#F5F4EC;
            padding:20px 28px;
            font-size:110%;
            line-height:1.8;
            color:black;">

<h2 style="text-align:center;">Mô tả dữ liệu</h2>

<p><b>Nguồn API:</b></p>
<ul>
    <li><b>orders:</b> Đơn hàng từ Shopee, TikTok Shop và Lazada. Mỗi dòng là một đơn hàng, gồm mã đơn, ngày đặt, sàn bán, cửa hàng, kho và trạng thái.</li>
    <li><b>order_items:</b> Chi tiết sản phẩm trong đơn hàng, gồm mã SKU, số lượng, giá bán, giảm giá, hoàn tiền và giá vốn. Một đơn có thể có nhiều dòng sản phẩm.</li>
    <li><b>payments:</b> Giao dịch thanh toán theo đơn, dùng đối soát số tiền dự kiến và thực nhận.</li>
    <li><b>warehouse_tasks:</b> Tác vụ đóng gói của nhân viên kho, gồm số sản phẩm xử lý, thời gian thực hiện và lỗi.</li>
    <li><b>ads_daily:</b> Kết quả quảng cáo TikTok theo ngày và video, gồm lượt hiển thị, lượt nhấp, chi phí, số đơn và doanh thu do nền tảng ghi nhận.</li>
</ul>

<p><b>Nguồn Excel nội bộ:</b></p>
<ul>
    <li><b>receipts:</b> Phiếu nhập kho, gồm ngày nhập, mã kho, SKU, số lượng và giá vốn một sản phẩm.</li>
    <li><b>attendance:</b> Chấm công từng nhân viên theo ngày, gồm bộ phận, giờ ca và giờ nghỉ.</li>
    <li><b>inventory_movements:</b> Sổ biến động số lượng kho, gồm tồn đầu kỳ, nhập hàng, xuất hàng và hàng hoàn. Dùng tính tồn kho theo từng SKU và kho.</li>
</ul>

<p><b>Nguồn Google Sheets:</b></p>
<ul>
    <li><b>products:</b> Danh mục sản phẩm, gồm SKU, tên sản phẩm, thương hiệu, ngành hàng, giá vốn và giá bán niêm yết.</li>
    <li><b>content:</b> Danh mục video, gồm người phụ trách, SKU quảng bá, ngày đăng, chủ đề và chi phí sản xuất.</li>
    <li><b>stock_counts:</b> Số lượng kiểm kê thực tế theo SKU, kho và ngày kiểm kê. Dùng so sánh với tồn trên sổ.</li>
</ul>

</div>

# 1. Import data từ nguồn Excel

In [3]:
from pathlib import Path

ROOT = Path.cwd()

excel_path = ROOT / "sources" / "excel" / "internal_operations.xlsx"

print("Thư mục đang làm việc:", ROOT)
print("Tìm thấy file Excel:", excel_path.exists())

Thư mục đang làm việc: /Users/hieutran/Downloads/DA_A_Z
Tìm thấy file Excel: True


In [4]:
import pandas as pd

tables = pd.read_excel(
    excel_path,
    sheet_name=None,
    dtype=str,
    keep_default_na=False
)

for name, df in tables.items():
    print(f"{name}: {len(df):,} dòng, {len(df.columns)} cột")

receipts: 1,200 dòng, 6 cột
attendance: 3,606 dòng, 8 cột
inventory_movements: 15,788 dòng, 8 cột


In [5]:
receipts = tables["receipts"]
attendance = tables["attendance"]
inventory_movements = tables["inventory_movements"]

display(receipts.head())
display(attendance.head())
display(inventory_movements.head())

,receipt_id,date,warehouse_id,sku,qty,unit_cost_vnd
0,RC0001,2026-01-16 00:00:00,WH02,SKU0001,77,238000
1,RC0002,2026-01-17 00:00:00,WH01,SKU0002,74,55000
2,RC0003,2026-01-18 00:00:00,WH02,SKU0003,40,52000
3,RC0004,2026-01-19 00:00:00,WH01,SKU0004,66,143000
4,RC0005,2026-01-20 00:00:00,WH02,SKU0005,71,55000


,attendance_id,date,employee_id,employee_name,department,paid_hours,break_hours,warehouse_id
0,ATT00001,2026-01-01 00:00:00,e001,Nhân viên 01,Kho,0,0,WH01
1,ATT00002,2026-01-01 00:00:00,e002,Nhân viên 02,Kho,8,0.5,WH02
2,ATT00003,2026-01-01 00:00:00,e003,Nhân viên 03,Kho,8,0.5,WH01
3,ATT00004,2026-01-01 00:00:00,e004,Nhân viên 04,Kho,7,0.5,WH02
4,ATT00005,2026-01-01 00:00:00,e005,Nhân viên 05,Kho,8,0.5,WH01


,movement_id,date,sku,warehouse_id,movement_type,qty_signed,order_id,unit_cost_vnd
0,MOV000001,2026-01-01 00:00:00,SKU0001,WH01,OPENING,60,,238000
1,MOV000002,2026-01-01 00:00:00,SKU0001,WH02,OPENING,60,,238000
2,MOV000003,2026-01-01 00:00:00,SKU0002,WH01,OPENING,60,,55000
3,MOV000004,2026-01-01 00:00:00,SKU0002,WH02,OPENING,60,,55000
4,MOV000005,2026-01-01 00:00:00,SKU0003,WH01,OPENING,60,,52000


#### kiểm tra kiểu dữ liệu

In [6]:
for name, df in tables.items():
    print(f"\n--- {name} ---")
    print(df.dtypes)


--- receipts ---
receipt_id       object
date             object
warehouse_id     object
sku              object
qty              object
unit_cost_vnd    object
dtype: object

--- attendance ---
attendance_id    object
date             object
employee_id      object
employee_name    object
department       object
paid_hours       object
break_hours      object
warehouse_id     object
dtype: object

--- inventory_movements ---
movement_id      object
date             object
sku              object
warehouse_id     object
movement_type    object
qty_signed       object
order_id         object
unit_cost_vnd    object
dtype: object


#### Kiểm tra dữ liệu thiếu và trùng

In [7]:
primary_keys = {
    "receipts": "receipt_id",
    "attendance": "attendance_id",
    "inventory_movements": "movement_id"
}

summary = []

for name, df in tables.items():
    key = primary_keys[name]

    # Tính cả ô null, chuỗi rỗng và chuỗi chỉ có khoảng trắng
    missing = df.apply(
        lambda col: col.isna() | col.str.strip().eq("")
    )

    summary.append({
        "table": name,
        "rows": len(df),
        "columns": len(df.columns),
        "exact_duplicates": int(df.duplicated().sum()),
        "duplicate_keys": int(df.duplicated(subset=[key]).sum()),
        "missing_cells": int(missing.sum().sum())
    })

quality_summary = pd.DataFrame(summary)

display(quality_summary)

,table,rows,columns,exact_duplicates,duplicate_keys,missing_cells
0,receipts,1200,6,0,0,0
1,attendance,3606,8,6,6,0
2,inventory_movements,15788,8,0,0,3600


#### Xem cụ thể các bản ghi chấm công trùng

In [8]:
duplicate_attendance = attendance[
    attendance.duplicated(
        subset=["attendance_id"],
        keep=False
    )
].sort_values("attendance_id")

display(duplicate_attendance)

,attendance_id,date,employee_id,employee_name,department,paid_hours,break_hours,warehouse_id
0,ATT00001,2026-01-01 00:00:00,e001,Nhân viên 01,Kho,0,0,WH01
3600,ATT00001,2026-01-01 00:00:00,e001,Nhân viên 01,Kho,0,0,WH01
1,ATT00002,2026-01-01 00:00:00,e002,Nhân viên 02,Kho,8,0.5,WH02
3601,ATT00002,2026-01-01 00:00:00,e002,Nhân viên 02,Kho,8,0.5,WH02
2,ATT00003,2026-01-01 00:00:00,e003,Nhân viên 03,Kho,8,0.5,WH01
3602,ATT00003,2026-01-01 00:00:00,e003,Nhân viên 03,Kho,8,0.5,WH01
3,ATT00004,2026-01-01 00:00:00,e004,Nhân viên 04,Kho,7,0.5,WH02
3603,ATT00004,2026-01-01 00:00:00,e004,Nhân viên 04,Kho,7,0.5,WH02
4,ATT00005,2026-01-01 00:00:00,e005,Nhân viên 05,Kho,8,0.5,WH01
3604,ATT00005,2026-01-01 00:00:00,e005,Nhân viên 05,Kho,8,0.5,WH01


#### Kiểm tra mã nhân viên không đúng định dang

In [9]:
employee_ids = attendance["employee_id"]

nonstandard_ids = attendance[
    employee_ids.ne(employee_ids.str.strip().str.upper())
]

display(nonstandard_ids)

,attendance_id,date,employee_id,employee_name,department,paid_hours,break_hours,warehouse_id
0,ATT00001,2026-01-01 00:00:00,e001,Nhân viên 01,Kho,0,0,WH01
1,ATT00002,2026-01-01 00:00:00,e002,Nhân viên 02,Kho,8,0.5,WH02
2,ATT00003,2026-01-01 00:00:00,e003,Nhân viên 03,Kho,8,0.5,WH01
3,ATT00004,2026-01-01 00:00:00,e004,Nhân viên 04,Kho,7,0.5,WH02
4,ATT00005,2026-01-01 00:00:00,e005,Nhân viên 05,Kho,8,0.5,WH01
5,ATT00006,2026-01-01 00:00:00,e006,Nhân viên 06,Kho,8,0.5,WH02
6,ATT00007,2026-01-01 00:00:00,e007,Nhân viên 07,Kho,8,0.5,WH01
7,ATT00008,2026-01-01 00:00:00,e008,Nhân viên 08,Kho,0,0,WH02
8,ATT00009,2026-01-01 00:00:00,e009,Nhân viên 09,Kho,8,0.5,WH01
9,ATT00010,2026-01-01 00:00:00,e010,Nhân viên 10,Kho,9,0.5,WH02


In [10]:
report_dir = ROOT / "reports"
report_dir.mkdir(exist_ok=True)

quality_summary.to_csv(
    report_dir / "excel_quality_before.csv",
    index=False,
    encoding="utf-8-sig"
)

### Clean Data

#### Tạo bản sao để giữ nguyên dữ liệu đã đọc

In [11]:
clean_tables = {
    name: df.copy()
    for name, df in tables.items()
}

#### Chuẩn hóa mã, xóa bản sao giống hoàn toàn và kiểm tra khóa trùng

In [12]:
cleaning_log = []

for name, df in clean_tables.items():
    before = len(df)

    # Chuẩn hóa các cột mã
    id_columns = [
        col for col in df.columns
        if col.endswith("_id") or col == "sku"
    ]

    for col in id_columns:
        df[col] = df[col].str.strip().str.upper()

    # Chỉ xóa các dòng giống hoàn toàn sau chuẩn hóa
    df = df.drop_duplicates().copy()

    key = primary_keys[name]

    # Dừng nếu cùng khóa nhưng nội dung vẫn khác nhau
    conflicts = df[df.duplicated(subset=[key], keep=False)]

    if not conflicts.empty:
        display(conflicts)
        raise ValueError(f"{name}: cần kiểm tra các bản ghi cùng khóa")

    assert df[key].str.strip().ne("").all(), f"{name}: thiếu khóa"

    clean_tables[name] = df

    cleaning_log.append({
        "table": name,
        "rows_before": before,
        "duplicates_removed": before - len(df),
        "rows_after": len(df)
    })

display(pd.DataFrame(cleaning_log))

,table,rows_before,duplicates_removed,rows_after
0,receipts,1200,0,1200
1,attendance,3606,6,3600
2,inventory_movements,15788,0,15788


#### Chuyển số và ngày về đúng kiểu

In [13]:
numeric_columns = {
    "receipts": ["qty", "unit_cost_vnd"],
    "attendance": ["paid_hours", "break_hours"],
    "inventory_movements": ["qty_signed", "unit_cost_vnd"]
}

for name, df in clean_tables.items():
    for col in numeric_columns[name]:
        df[col] = pd.to_numeric(
            df[col]
                .astype("string")
                .str.replace(",", "", regex=False)
                .str.strip(),
            errors="raise"
        )

    df["date"] = pd.to_datetime(df["date"], errors="raise")

print("Đã chuyển các cột số và ngày thành công.")

Đã chuyển các cột số và ngày thành công.


In [14]:
for name, df in clean_tables.items():
    print(f"\nBảng: {name}")
    print(df[numeric_columns[name] + ["date"]].dtypes)


Bảng: receipts
qty                       Int64
unit_cost_vnd             Int64
date             datetime64[ns]
dtype: object

Bảng: attendance
paid_hours              Int64
break_hours           Float64
date           datetime64[ns]
dtype: object

Bảng: inventory_movements
qty_signed                Int64
unit_cost_vnd             Int64
date             datetime64[ns]
dtype: object


In [15]:
for name, df in clean_tables.items():
    columns_to_check = numeric_columns[name] + ["date"]

    print(f"\nBảng: {name}")
    print(df[columns_to_check].isna().sum())


Bảng: receipts
qty              0
unit_cost_vnd    0
date             0
dtype: int64

Bảng: attendance
paid_hours     0
break_hours    0
date           0
dtype: int64

Bảng: inventory_movements
qty_signed       0
unit_cost_vnd    0
date             0
dtype: int64


#### Kiểm tra giờ công

In [16]:
attendance = clean_tables["attendance"]

invalid_hours = attendance[
    (attendance["paid_hours"] < 0)
    | (attendance["break_hours"] < 0)
    | (attendance["break_hours"] > attendance["paid_hours"])
]

print("Số dòng có giờ công không hợp lệ:", len(invalid_hours))
display(invalid_hours)

Số dòng có giờ công không hợp lệ: 0


,attendance_id,date,employee_id,employee_name,department,paid_hours,break_hours,warehouse_id


#### Tạo cột giờ khả dụng

In [17]:
clean_tables["attendance"]["available_hours"] = (
    clean_tables["attendance"]["paid_hours"]
    - clean_tables["attendance"]["break_hours"]
)

display(
    clean_tables["attendance"][
        [
            "employee_id",
            "date",
            "paid_hours",
            "break_hours",
            "available_hours"
        ]
    ].head(10)
)

,employee_id,date,paid_hours,break_hours,available_hours
0,E001,2026-01-01,0,0.0,0.0
1,E002,2026-01-01,8,0.5,7.5
2,E003,2026-01-01,8,0.5,7.5
3,E004,2026-01-01,7,0.5,6.5
4,E005,2026-01-01,8,0.5,7.5
5,E006,2026-01-01,8,0.5,7.5
6,E007,2026-01-01,8,0.5,7.5
7,E008,2026-01-01,0,0.0,0.0
8,E009,2026-01-01,8,0.5,7.5
9,E010,2026-01-01,9,0.5,8.5


<div style="border:3px solid black;
            border-radius:10px;
            background-color:#FBE5D9;
            padding:20px 28px;
            font-size:110%;
            line-height:1.8;
            color:black;">

Kiểm tra số lượng nhập/xuất kho

Quy tắc:
- OPENING (tồn đầu kỳ):	    qty_signed > 0
- RECEIPT (nhập kho):	    qty_signed > 0
- RETURN  (hàng hoàn về):	qty_signed > 0
- SHIP    (xuất kho):    	qty_signed < 0
</div> 

In [18]:
movements = clean_tables["inventory_movements"]

positive_types = ["OPENING", "RECEIPT", "RETURN"]
allowed_types = positive_types + ["SHIP"]

invalid_movements = movements[
    (
        movements["movement_type"].isin(positive_types)
        & (movements["qty_signed"] <= 0)
    )
    | (
        movements["movement_type"].eq("SHIP")
        & (movements["qty_signed"] >= 0)
    )
    | (~movements["movement_type"].isin(allowed_types))
]

print("Số dòng sai dấu hoặc sai loại:", len(invalid_movements))
display(invalid_movements)

Số dòng sai dấu hoặc sai loại: 0


,movement_id,date,sku,warehouse_id,movement_type,qty_signed,order_id,unit_cost_vnd


#### kiểm tra số lượng và giá vốn trong phiếu nhập kho


Quy tắc:
- qty phải lớn hơn 0 và là số nguyên.
- unit_cost_vnd phải lớn hơn 0.
- Hai cột này không được thiếu dữ liệu.

In [19]:
receipts = clean_tables["receipts"]

invalid_receipts = receipts[
    receipts["qty"].isna()
    | receipts["unit_cost_vnd"].isna()
    | (receipts["qty"] <= 0)
    | (receipts["qty"] % 1 != 0)
    | (receipts["unit_cost_vnd"] <= 0)
]

print("Số dòng nhập kho không hợp lệ:", len(invalid_receipts))
display(invalid_receipts)

Số dòng nhập kho không hợp lệ: 0


,receipt_id,date,warehouse_id,sku,qty,unit_cost_vnd


#### Xuất data (3 file excel đã clean)

In [20]:
# Tạo thư mục lưu dữ liệu sạch
clean_dir = ROOT / "output" / "clean"
clean_dir.mkdir(parents=True, exist_ok=True)

# Lưu đúng 3 bảng Excel
for name in ["receipts", "attendance", "inventory_movements"]:
    file_path = clean_dir / f"{name}.csv"

    clean_tables[name].to_csv(
        file_path,
        index=False,
        encoding="utf-8-sig",
        date_format="%Y-%m-%d"
    )

    print(f"Đã lưu: {file_path.name} — {len(clean_tables[name]):,} dòng")

# Lưu log làm sạch
reports_dir = ROOT / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame(cleaning_log).to_csv(
    reports_dir / "cleaning_log_excel.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nThư mục dữ liệu sạch:", clean_dir.resolve())
print("Đã lưu log làm sạch.")

Đã lưu: receipts.csv — 1,200 dòng
Đã lưu: attendance.csv — 3,600 dòng
Đã lưu: inventory_movements.csv — 15,788 dòng

Thư mục dữ liệu sạch: /Users/hieutran/Downloads/DA_A_Z/output/clean
Đã lưu log làm sạch.


# 2. Import data từ nguồn gg sheet

In [21]:
import pandas as pd

SHEET_ID = "171bLzYnVB1eKxKPTAIJMcwqpOK1lMo48FoNStVRKm-Y"

sheet_gids = {
    "products": "1771824124",
    "content": "312843898",
    "stock_counts": "980507028"
}

expected_shapes = {
    "products": (1200, 8),
    "content": (1200, 7),
    "stock_counts": (2400, 5)
}

sheets_link_tables = {}

for name, gid in sheet_gids.items():
    url = (
        f"https://docs.google.com/spreadsheets/d/{SHEET_ID}"
        f"/export?format=csv&gid={gid}"
    )

    print(f"Đang đọc: {name}")

    df = pd.read_csv(
        url,
        dtype=str,
        keep_default_na=False
    )

    assert df.shape == expected_shapes[name], (
        f"{name}: kích thước không đúng, nhận được {df.shape}"
    )

    sheets_link_tables[name] = df
    print(f"Đã đọc: {len(df):,} dòng, {len(df.columns)} cột")

print("\nĐã đọc đủ 3 bảng qua link CSV.")

Đang đọc: products
Đã đọc: 1,200 dòng, 8 cột
Đang đọc: content
Đã đọc: 1,200 dòng, 7 cột
Đang đọc: stock_counts
Đã đọc: 2,400 dòng, 5 cột

Đã đọc đủ 3 bảng qua link CSV.


#### Tạo bản sao ba bảng Sheets để clean data

In [22]:
from datetime import datetime

run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

raw_dir = ROOT / "output" / "raw" / f"sheets_link_{run_id}"
staging_dir = ROOT / "output" / "staging"

raw_dir.mkdir(parents=True, exist_ok=True)
staging_dir.mkdir(parents=True, exist_ok=True)

for name in ["products", "content", "stock_counts"]:
    df = sheets_link_tables[name]

    # Lưu dữ liệu chưa làm sạch theo từng lần lấy
    df.to_csv(
        raw_dir / f"{name}.csv",
        index=False,
        encoding="utf-8-sig"
    )

    # Lưu bản dùng làm đầu vào cho bước clean
    df.to_csv(
        staging_dir / f"{name}.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Đã lưu {name}: {len(df):,} dòng")

print("\nThư mục raw:", raw_dir.resolve())
print("Thư mục staging:", staging_dir.resolve())

Đã lưu products: 1,200 dòng
Đã lưu content: 1,200 dòng
Đã lưu stock_counts: 2,400 dòng

Thư mục raw: /Users/hieutran/Downloads/DA_A_Z/output/raw/sheets_link_20261014_171357_750343
Thư mục staging: /Users/hieutran/Downloads/DA_A_Z/output/staging


In [23]:
clean_sheet_tables = {
    name: df.copy()
    for name, df in sheets_link_tables.items()
}

for name, df in clean_sheet_tables.items():
    print(f"\nBảng: {name} — {len(df):,} dòng")
    display(df.head(3))


Bảng: products — 1,200 dòng


,sku,product_name,brand,category,standard_cost_vnd,list_price_vnd,reorder_point,lead_days
0,SKU0001,Sản phẩm gia dụng 0001,AnHome,Bếp,238000,260000,20,7
1,SKU0002,Sản phẩm gia dụng 0002,BepViet,Vệ sinh,55000,75000,20,7
2,SKU0003,Sản phẩm gia dụng 0003,NhaXinh,Lưu trữ,52000,89000,20,7



Bảng: content — 1,200 dòng


,video_id,creator_id,creator_name,sku,publish_date,production_cost_vnd,topic
0,VID0001,,Nhân viên 21,SKU0001,2026-01-01,100000,Demo
1,VID0002,,Nhân viên 22,SKU0002,2026-01-02,100000,Review
2,VID0003,,Nhân viên 23,SKU0003,2026-01-03,100000,Mẹo sử dụng



Bảng: stock_counts — 2,400 dòng


,count_id,count_date,sku,warehouse_id,counted_qty
0,COUNT00001,2026-03-31,SKU0001,WH01,59
1,COUNT00002,2026-03-31,SKU0001,WH02,130
2,COUNT00003,2026-03-31,SKU0002,WH01,129


#### Kiểm tra missing value

In [24]:
sheet_primary_keys = {
    "products": "sku",
    "content": "video_id",
    "stock_counts": "count_id"
}

for name, df in clean_sheet_tables.items():
    key = sheet_primary_keys[name]

    # Tính cả ô rỗng và ô chỉ chứa khoảng trắng
    blank_cells = df.apply(
        lambda col: col.isna()
        | col.astype("string").str.strip().eq("")
    )

    print(f"\nBảng: {name}")
    print("Số dòng trùng khóa:", df.duplicated(subset=[key]).sum())
    print("Số ô thiếu theo cột:")
    print(blank_cells.sum())


Bảng: products
Số dòng trùng khóa: 0
Số ô thiếu theo cột:
sku                  0
product_name         0
brand                0
category             0
standard_cost_vnd    0
list_price_vnd       0
reorder_point        0
lead_days            0
dtype: Int64

Bảng: content
Số dòng trùng khóa: 0
Số ô thiếu theo cột:
video_id               0
creator_id             8
creator_name           0
sku                    0
publish_date           0
production_cost_vnd    0
topic                  0
dtype: Int64

Bảng: stock_counts
Số dòng trùng khóa: 0
Số ô thiếu theo cột:
count_id        0
count_date      0
sku             0
warehouse_id    0
counted_qty     0
dtype: Int64


<div style="border:3px solid black;
            border-radius:10px;
            background-color:#FBE5D9;
            padding:20px 28px;
            font-size:110%;
            line-height:1.8;
            color:black;">

→ Nhận xét: Ở bảng content, cột creator_id bị thiếu 8 bản ghi

Hướng xử lý: Điền thành "Unknow". Không dùng tên (Creator_name) để đoán mã người phụ trách vì tên có thể trùng

</div>

In [25]:
content = clean_sheet_tables["content"]

missing_creator = (
    content["creator_id"].isna()
    | content["creator_id"].eq("")
)

print("Số video thiếu creator_id:", missing_creator.sum())

content.loc[missing_creator, "creator_id"] = "UNKNOWN"

display(
    content.loc[
        missing_creator,
        ["video_id", "creator_id", "creator_name"]
    ]
)

Số video thiếu creator_id: 8


,video_id,creator_id,creator_name
0,VID0001,UNKNOWN,Nhân viên 21
1,VID0002,UNKNOWN,Nhân viên 22
2,VID0003,UNKNOWN,Nhân viên 23
3,VID0004,UNKNOWN,Nhân viên 24
4,VID0005,UNKNOWN,Nhân viên 25
5,VID0006,UNKNOWN,Nhân viên 26
6,VID0007,UNKNOWN,Nhân viên 27
7,VID0008,UNKNOWN,Nhân viên 28


#### Chuẩn hóa các cột mã trong ba bảng

In [26]:
for name, df in clean_sheet_tables.items():
    id_columns = [
        col for col in df.columns
        if col.endswith("_id") or col == "sku"
    ]

    for col in id_columns:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .str.upper()
        )

    print(f"{name}: đã chuẩn hóa {id_columns}")

display(
    clean_sheet_tables["content"][
        ["video_id", "creator_id", "sku"]
    ].head(10)
)

products: đã chuẩn hóa ['sku']
content: đã chuẩn hóa ['video_id', 'creator_id', 'sku']
stock_counts: đã chuẩn hóa ['count_id', 'sku', 'warehouse_id']


,video_id,creator_id,sku
0,VID0001,UNKNOWN,SKU0001
1,VID0002,UNKNOWN,SKU0002
2,VID0003,UNKNOWN,SKU0003
3,VID0004,UNKNOWN,SKU0004
4,VID0005,UNKNOWN,SKU0005
5,VID0006,UNKNOWN,SKU0006
6,VID0007,UNKNOWN,SKU0007
7,VID0008,UNKNOWN,SKU0008
8,VID0009,E029,SKU0009
9,VID0010,E030,SKU0010


#### Kiểm tra kiểu dữ liệu

In [27]:
for name, df in clean_sheet_tables.items():
    print(f"\nBảng: {name}")
    print(df.dtypes)


Bảng: products
sku                  string[python]
product_name                 object
brand                        object
category                     object
standard_cost_vnd            object
list_price_vnd               object
reorder_point                object
lead_days                    object
dtype: object

Bảng: content
video_id               string[python]
creator_id             string[python]
creator_name                   object
sku                    string[python]
publish_date                   object
production_cost_vnd            object
topic                          object
dtype: object

Bảng: stock_counts
count_id        string[python]
count_date              object
sku             string[python]
warehouse_id    string[python]
counted_qty             object
dtype: object


#### Chuyển đổi kiểu dữ liệu

In [28]:
sheet_numeric_columns = {
    "products": [
        "standard_cost_vnd", "list_price_vnd",
        "reorder_point", "lead_days"
    ],
    "content": ["production_cost_vnd"],
    "stock_counts": ["counted_qty"]
}

sheet_date_columns = {
    "products": [],
    "content": ["publish_date"],
    "stock_counts": ["count_date"]
}

for name, df in clean_sheet_tables.items():
    for col in sheet_numeric_columns[name]:
        df[col] = pd.to_numeric(
            df[col]
                .astype("string")
                .str.replace(",", "", regex=False)
                .str.strip(),
            errors="raise"
        )

    for col in sheet_date_columns[name]:
        df[col] = pd.to_datetime(
            df[col],
            format="%Y-%m-%d",
            errors="raise"
        )

    columns = sheet_numeric_columns[name] + sheet_date_columns[name]

    print(f"\nBảng: {name}")
    print(df[columns].dtypes)


Bảng: products
standard_cost_vnd    Int64
list_price_vnd       Int64
reorder_point        Int64
lead_days            Int64
dtype: object

Bảng: content
production_cost_vnd             Int64
publish_date           datetime64[ns]
dtype: object

Bảng: stock_counts
counted_qty             Int64
count_date     datetime64[ns]
dtype: object


#### Kiểm tra Key

In [29]:
sheet_primary_keys = {
    "products": "sku",
    "content": "video_id",
    "stock_counts": "count_id"
}

for name, df in clean_sheet_tables.items():
    key = sheet_primary_keys[name]

    missing_key = (
        df[key].isna()
        | df[key].astype("string").str.strip().eq("")
    )

    duplicate_key = df.duplicated(subset=[key], keep=False)

    print(f"\nBảng: {name}")
    print("Số dòng thiếu khóa:", missing_key.sum())
    print("Số dòng có khóa trùng:", duplicate_key.sum())

    if missing_key.any() or duplicate_key.any():
        display(df.loc[missing_key | duplicate_key])


Bảng: products
Số dòng thiếu khóa: 0
Số dòng có khóa trùng: 0

Bảng: content
Số dòng thiếu khóa: 0
Số dòng có khóa trùng: 0

Bảng: stock_counts
Số dòng thiếu khóa: 0
Số dòng có khóa trùng: 0


<div style="border:3px solid black;
            border-radius:10px;
            background-color:#FBE5D9;
            padding:20px 28px;
            font-size:110%;
            line-height:1.8;
            color:black;">
    
<p><b>Kiểm tra SKU giữa các bảng</b></p>
<ul>


Quy tắc: mọi sku trong content và stock_counts phải tồn tại trong danh mục products

Kiểm tra này giúp tránh mất dữ liệu khi JOIN với products

</div>

In [36]:
valid_skus = set(clean_sheet_tables["products"]["sku"].dropna())

for name in ["content", "stock_counts"]:
    df = clean_sheet_tables[name]

    invalid_skus = df.loc[
        ~df["sku"].isin(valid_skus)
    ]

    print(f"{name}: {len(invalid_skus)} dòng có SKU ngoài danh mục")

    if not invalid_skus.empty:
        display(invalid_skus)

content: 0 dòng có SKU ngoài danh mục
stock_counts: 0 dòng có SKU ngoài danh mục


#### kiểm tra các cột số có bị thiếu hoặc âm không

##### Trong bài này, giá vốn và giá niêm yết phải lớn hơn 0; ngưỡng tồn, thời gian chờ, chi phí video và số lượng kiểm kê được phép bằng 0, nhưng không được âm.

In [33]:
for name, df in clean_sheet_tables.items():
    for col in sheet_numeric_columns[name]:
        must_be_positive = col in [
            "standard_cost_vnd", "list_price_vnd"
        ]

        invalid = df[col].isna()

        if must_be_positive:
            invalid = invalid | (df[col] <= 0)
        else:
            invalid = invalid | (df[col] < 0)

        print(f"{name}.{col}: {invalid.sum()} dòng lỗi")

        if invalid.any():
            display(df.loc[invalid])

products.standard_cost_vnd: 0 dòng lỗi
products.list_price_vnd: 0 dòng lỗi
products.reorder_point: 0 dòng lỗi
products.lead_days: 0 dòng lỗi
content.production_cost_vnd: 0 dòng lỗi
stock_counts.counted_qty: 0 dòng lỗi


#### kiểm tra ngày và creator_id

In [37]:
for name, col in {
    "content": "publish_date",
    "stock_counts": "count_date"
}.items():
    df = clean_sheet_tables[name]

    invalid = (
        df[col].isna()
        | (df[col] < pd.Timestamp("2026-01-01"))
        | (df[col] >= pd.Timestamp("2026-04-01"))
    )

    print(f"{name}.{col}: {invalid.sum()} dòng lỗi")

creator = clean_sheet_tables["content"]["creator_id"]
missing_creator = creator.isna() | creator.str.strip().eq("")

print("creator_id còn thiếu:", missing_creator.sum())
print("Số video thuộc UNKNOWN:", creator.eq("UNKNOWN").sum())

content.publish_date: 0 dòng lỗi
stock_counts.count_date: 0 dòng lỗi
creator_id còn thiếu: 0
Số video thuộc UNKNOWN: 8


#### Xuất data (3 bảng sheet đã clean)

In [38]:
clean_dir = ROOT / "output" / "clean"
clean_dir.mkdir(parents=True, exist_ok=True)

for name in ["products", "content", "stock_counts"]:
    df = clean_sheet_tables[name]
    file_path = clean_dir / f"{name}.csv"

    df.to_csv(
        file_path,
        index=False,
        encoding="utf-8-sig",
        date_format="%Y-%m-%d"
    )

    print(f"Đã lưu {file_path.name}: {len(df):,} dòng")

print("\nThư mục lưu:", clean_dir.resolve())

Đã lưu products.csv: 1,200 dòng
Đã lưu content.csv: 1,200 dòng
Đã lưu stock_counts.csv: 2,400 dòng

Thư mục lưu: /Users/hieutran/Downloads/DA_A_Z/output/clean


# 3. Import data từ nguồn API

In [39]:
print(ROOT.resolve())
print("Có file API:", (ROOT / "tools" / "mock_api.py").is_file())

/Users/hieutran/Downloads/DA_A_Z
Có file API: True


#### Lấy data từ sàn Shopee

In [40]:
import requests
import pandas as pd

BASE_URL = "http://127.0.0.1:8000"
HEADERS = {
    "Authorization": "Bearer training-token"
}

response = requests.get(
    BASE_URL + "/shopee/orders",
    headers=HEADERS,
    params={"page": 1, "page_size": 5},
    timeout=(5, 30)
)

response.raise_for_status()
payload = response.json()

print("HTTP status:", response.status_code)
print("Tổng dòng nguồn Shopee:", payload["total"])

shopee_sample = pd.DataFrame(
    payload["response"]["order_list"]
)

display(shopee_sample)

HTTP status: 200
Tổng dòng nguồn Shopee: 2015


,order_id,order_date,updated_at,channel,shop_id,warehouse_id,status,video_id,platform_fee_vnd,shipping_cost_vnd
0,O000003,2026-02-06,2026-02-10T12:00:00+07:00,Shopee,SP01,WH01,completed,,109440,20000
1,O000006,2026-03-18,2026-03-22T12:00:00+07:00,Shopee,SP01,WH02,completed,,98640,20000
2,O000007,2026-02-10,2026-02-14T12:00:00+07:00,Shopee,SP01,WH01,completed,,110040,20000
3,O000008,2026-01-09,2026-01-13T12:00:00+07:00,Shopee,SP01,WH02,completed,,151560,20000
4,O000010,2026-02-12,2026-02-16T12:00:00+07:00,Shopee,SP01,WH01,completed,,214440,20000


In [41]:
def parse_page(payload, style):
    if style == "shopee":
        data = payload["response"]
        return data["order_list"], data["more"]

    if style == "tiktok":
        assert payload["code"] == 0
        data = payload["data"]
        return data["orders"], data["has_more"]

    if style == "lazada":
        assert payload["code"] == "0"
        data = payload["result"]
        return data["orders"], data["has_next"]

    if style == "generic":
        return (
            payload["data"],
            payload["pagination"]["has_next"]
        )

    raise ValueError(f"Kiểu phản hồi không hỗ trợ: {style}")


# Thử với phản hồi Shopee vừa lấy
rows, has_more = parse_page(payload, "shopee")

print("Số bản ghi của trang:", len(rows))
print("Còn trang tiếp theo:", has_more)

Số bản ghi của trang: 5
Còn trang tiếp theo: True


In [42]:
import json
from datetime import datetime

api_run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
api_raw_dir = ROOT / "output" / "raw" / f"api_{api_run_id}"
api_raw_dir.mkdir(parents=True, exist_ok=True)


def fetch_api(path, style="generic"):
    all_rows = []
    page = 1
    expected_total = None

    while True:
        response = requests.get(
            BASE_URL + path,
            headers=HEADERS,
            params={"page": page, "page_size": 200},
            timeout=(5, 30)
        )
        response.raise_for_status()
        page_payload = response.json()

        # Lưu phản hồi gốc của từng trang
        tag = path.strip("/").replace("/", "_")
        raw_path = api_raw_dir / f"{tag}_{page:04d}.json"

        raw_path.write_text(
            json.dumps(page_payload, ensure_ascii=False, indent=2),
            encoding="utf-8"
        )

        if expected_total is None:
            expected_total = page_payload["total"]

        assert page_payload["total"] == expected_total, (
            "Tổng nguồn thay đổi trong lúc lấy dữ liệu"
        )

        rows, has_more = parse_page(page_payload, style)
        all_rows.extend(rows)

        print(f"{tag} — trang {page}: {len(rows)} dòng")

        if not has_more:
            break

        assert rows, "Trang rỗng nhưng API báo còn trang"
        page += 1

    assert len(all_rows) == expected_total, (
        f"Lấy được {len(all_rows)} dòng, nguồn báo {expected_total}"
    )

    return pd.DataFrame(all_rows)


print("Đã tạo hàm fetch_api.")

Đã tạo hàm fetch_api.


In [43]:
shopee_orders = fetch_api("/shopee/orders", "shopee")

print("\nTổng dòng Shopee:", len(shopee_orders))
display(shopee_orders.head(3))

shopee_orders — trang 1: 200 dòng
shopee_orders — trang 2: 200 dòng
shopee_orders — trang 3: 200 dòng
shopee_orders — trang 4: 200 dòng
shopee_orders — trang 5: 200 dòng
shopee_orders — trang 6: 200 dòng
shopee_orders — trang 7: 200 dòng
shopee_orders — trang 8: 200 dòng
shopee_orders — trang 9: 200 dòng
shopee_orders — trang 10: 200 dòng
shopee_orders — trang 11: 15 dòng

Tổng dòng Shopee: 2015


,order_id,order_date,updated_at,channel,shop_id,warehouse_id,status,video_id,platform_fee_vnd,shipping_cost_vnd
0,O000003,2026-02-06,2026-02-10T12:00:00+07:00,Shopee,SP01,WH01,completed,,109440,20000
1,O000006,2026-03-18,2026-03-22T12:00:00+07:00,Shopee,SP01,WH02,completed,,98640,20000
2,O000007,2026-02-10,2026-02-14T12:00:00+07:00,Shopee,SP01,WH01,completed,,110040,20000


#### Lấy data từ sàn Tiktok và Lazada

In [44]:
tiktok_orders = fetch_api("/tiktok-shop/orders", "tiktok")
lazada_orders = fetch_api("/lazada/orders", "lazada")

api_tables = {}

api_tables["orders"] = pd.concat(
    [shopee_orders, tiktok_orders, lazada_orders],
    ignore_index=True
)

print("\nSố dòng theo sàn:")
print(api_tables["orders"]["channel"].value_counts())

print("\nTổng dòng orders raw:", len(api_tables["orders"]))

assert len(api_tables["orders"]) == 6015, (
    "Tổng orders raw chưa khớp 6.015 dòng"
)

tiktok-shop_orders — trang 1: 200 dòng
tiktok-shop_orders — trang 2: 200 dòng
tiktok-shop_orders — trang 3: 200 dòng
tiktok-shop_orders — trang 4: 200 dòng
tiktok-shop_orders — trang 5: 200 dòng
tiktok-shop_orders — trang 6: 200 dòng
tiktok-shop_orders — trang 7: 200 dòng
tiktok-shop_orders — trang 8: 200 dòng
tiktok-shop_orders — trang 9: 200 dòng
tiktok-shop_orders — trang 10: 200 dòng
tiktok-shop_orders — trang 11: 7 dòng
lazada_orders — trang 1: 200 dòng
lazada_orders — trang 2: 200 dòng
lazada_orders — trang 3: 200 dòng
lazada_orders — trang 4: 200 dòng
lazada_orders — trang 5: 200 dòng
lazada_orders — trang 6: 200 dòng
lazada_orders — trang 7: 200 dòng
lazada_orders — trang 8: 200 dòng
lazada_orders — trang 9: 200 dòng
lazada_orders — trang 10: 193 dòng

Số dòng theo sàn:
channel
Shopee         2015
TikTok Shop    2007
Lazada         1993
Name: count, dtype: int64

Tổng dòng orders raw: 6015


#### Lấy data nội bộ

In [45]:
api_endpoints = {
    "order_items": "/oms/order-items",
    "payments": "/finance/payments",
    "warehouse_tasks": "/warehouse/tasks",
    "ads_daily": "/tiktok-ads/reports"
}

for name, path in api_endpoints.items():
    api_tables[name] = fetch_api(path, "generic")
    print(f"\nĐã lấy {name}: {len(api_tables[name]):,} dòng\n")

print("Số dòng của 5 bảng API:")
for name, df in api_tables.items():
    print(f"{name}: {len(df):,} dòng")

oms_order-items — trang 1: 200 dòng
oms_order-items — trang 2: 200 dòng
oms_order-items — trang 3: 200 dòng
oms_order-items — trang 4: 200 dòng
oms_order-items — trang 5: 200 dòng
oms_order-items — trang 6: 200 dòng
oms_order-items — trang 7: 200 dòng
oms_order-items — trang 8: 200 dòng
oms_order-items — trang 9: 200 dòng
oms_order-items — trang 10: 200 dòng
oms_order-items — trang 11: 200 dòng
oms_order-items — trang 12: 200 dòng
oms_order-items — trang 13: 200 dòng
oms_order-items — trang 14: 200 dòng
oms_order-items — trang 15: 200 dòng
oms_order-items — trang 16: 200 dòng
oms_order-items — trang 17: 200 dòng
oms_order-items — trang 18: 200 dòng
oms_order-items — trang 19: 200 dòng
oms_order-items — trang 20: 200 dòng
oms_order-items — trang 21: 200 dòng
oms_order-items — trang 22: 200 dòng
oms_order-items — trang 23: 200 dòng
oms_order-items — trang 24: 200 dòng
oms_order-items — trang 25: 200 dòng
oms_order-items — trang 26: 200 dòng
oms_order-items — trang 27: 200 dòng
oms_order-

#### Tạo bản sao trước khi clean data

In [46]:
staging_dir = ROOT / "output" / "staging"
staging_dir.mkdir(parents=True, exist_ok=True)

for name in [
    "orders",
    "order_items",
    "payments",
    "warehouse_tasks",
    "ads_daily"
]:
    df = api_tables[name]

    df.to_csv(
        staging_dir / f"{name}.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Đã lưu {name}.csv: {len(df):,} dòng")

print("\nThư mục staging:", staging_dir.resolve())

Đã lưu orders.csv: 6,015 dòng
Đã lưu order_items.csv: 12,105 dòng
Đã lưu payments.csv: 6,000 dòng
Đã lưu warehouse_tasks.csv: 5,530 dòng
Đã lưu ads_daily.csv: 6,000 dòng

Thư mục staging: /Users/hieutran/Downloads/DA_A_Z/output/staging


In [47]:
clean_api_tables = {
    name: df.copy()
    for name, df in api_tables.items()
}

for name, df in clean_api_tables.items():
    print(f"\nBảng: {name} — {len(df):,} dòng")
    print(df.dtypes)


Bảng: orders — 6,015 dòng
order_id             object
order_date           object
updated_at           object
channel              object
shop_id              object
warehouse_id         object
status               object
video_id             object
platform_fee_vnd      int64
shipping_cost_vnd     int64
dtype: object

Bảng: order_items — 12,105 dòng
order_item_id     object
order_id          object
sku               object
qty                int64
unit_price_vnd     int64
discount_vnd       int64
refund_vnd         int64
unit_cost_vnd      int64
dtype: object

Bảng: payments — 6,000 dòng
transaction_id         object
order_id               object
settlement_date        object
net_sales_vnd           int64
platform_fee_vnd        int64
shipping_cost_vnd       int64
expected_payout_vnd     int64
actual_payout_vnd       int64
dtype: object

Bảng: warehouse_tasks — 5,530 dòng
task_id             object
order_id            object
date                object
employee_id         object
wareh

#### kiểm tra khóa trùng và dữ liệu thiếu

In [48]:
api_primary_keys = {
    "orders": "order_id",
    "order_items": "order_item_id",
    "payments": "transaction_id",
    "warehouse_tasks": "task_id",
    "ads_daily": "ad_record_id"
}

for name, df in clean_api_tables.items():
    key = api_primary_keys[name]

    missing = df.apply(
        lambda col: col.isna()
        | col.astype("string").str.strip().eq("")
    )

    print(f"\nBảng: {name}")
    print("Số dòng:", len(df))
    print("Số dòng trùng hoàn toàn:", df.duplicated().sum())
    print("Số dòng trùng khóa vượt bản đầu:", df.duplicated(key).sum())
    print("Số dòng thiếu khóa:", missing[key].sum())
    print("Số ô thiếu theo cột:")
    print(missing.sum())


Bảng: orders
Số dòng: 6015
Số dòng trùng hoàn toàn: 15
Số dòng trùng khóa vượt bản đầu: 15
Số dòng thiếu khóa: 0
Số ô thiếu theo cột:
order_id                0
order_date              0
updated_at              0
channel                 0
shop_id                 0
warehouse_id            0
status                  0
video_id             5102
platform_fee_vnd        0
shipping_cost_vnd       0
dtype: Int64

Bảng: order_items
Số dòng: 12105
Số dòng trùng hoàn toàn: 0
Số dòng trùng khóa vượt bản đầu: 0
Số dòng thiếu khóa: 0
Số ô thiếu theo cột:
order_item_id     0
order_id          0
sku               0
qty               0
unit_price_vnd    0
discount_vnd      0
refund_vnd        0
unit_cost_vnd     0
dtype: Int64

Bảng: payments
Số dòng: 6000
Số dòng trùng hoàn toàn: 0
Số dòng trùng khóa vượt bản đầu: 0
Số dòng thiếu khóa: 0
Số ô thiếu theo cột:
transaction_id         0
order_id               0
settlement_date        0
net_sales_vnd          0
platform_fee_vnd       0
shipping_cost_vnd   

#### Chuẩn hóa mã và xóa bản sao giống hoàn toàn

In [49]:
api_cleaning_log = []

for name, df in clean_api_tables.items():
    df = df.copy()
    before = len(df)

    id_columns = [
        col for col in df.columns
        if col.endswith("_id") or col == "sku"
    ]

    for col in id_columns:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .str.upper()
        )

    # Chỉ xóa dòng giống hoàn toàn sau chuẩn hóa
    df = df.drop_duplicates().copy()

    key = api_primary_keys[name]

    conflicts = df.loc[
        df.duplicated(subset=[key], keep=False)
    ]

    if not conflicts.empty:
        display(conflicts)
        raise ValueError(f"{name}: cùng khóa nhưng khác nội dung")

    missing_key = df[key].isna() | df[key].eq("")
    assert not missing_key.any(), f"{name}: thiếu khóa"

    clean_api_tables[name] = df

    api_cleaning_log.append({
        "table": name,
        "rows_before": before,
        "duplicates_removed": before - len(df),
        "rows_after": len(df)
    })

display(pd.DataFrame(api_cleaning_log))

,table,rows_before,duplicates_removed,rows_after
0,orders,6015,15,6000
1,order_items,12105,0,12105
2,payments,6000,0,6000
3,warehouse_tasks,5530,0,5530
4,ads_daily,6000,0,6000


#### Chuẩn hoá các cột số và ngày

In [50]:
api_numeric_columns = {
    "orders": [
        "platform_fee_vnd", "shipping_cost_vnd"
    ],
    "order_items": [
        "qty", "unit_price_vnd", "discount_vnd",
        "refund_vnd", "unit_cost_vnd"
    ],
    "payments": [
        "net_sales_vnd", "platform_fee_vnd",
        "shipping_cost_vnd", "expected_payout_vnd",
        "actual_payout_vnd"
    ],
    "warehouse_tasks": [
        "units", "duration_minutes", "error_flag"
    ],
    "ads_daily": [
        "impressions", "clicks", "spend_vnd",
        "platform_attributed_orders",
        "platform_attributed_revenue_vnd"
    ]
}

api_date_columns = {
    "orders": ["order_date"],
    "order_items": [],
    "payments": ["settlement_date"],
    "warehouse_tasks": ["date"],
    "ads_daily": ["date"]
}

for name, df in clean_api_tables.items():
    for col in api_numeric_columns[name]:
        df[col] = pd.to_numeric(
            df[col]
                .astype("string")
                .str.replace(",", "", regex=False)
                .str.strip(),
            errors="raise"
        )

    for col in api_date_columns[name]:
        df[col] = pd.to_datetime(
            df[col],
            format="%Y-%m-%d",
            errors="raise"
        )

# Thời điểm cập nhật có múi giờ: xử lý riêng
clean_api_tables["orders"]["updated_at"] = pd.to_datetime(
    clean_api_tables["orders"]["updated_at"],
    utc=True,
    errors="raise"
).dt.tz_convert("Asia/Ho_Chi_Minh")

print("Đã chuyển kiểu số, ngày và thời điểm cập nhật.")

Đã chuyển kiểu số, ngày và thời điểm cập nhật.


In [51]:
for name, df in clean_api_tables.items():
    print(f"\nBảng: {name}")
    print(df.dtypes)


Bảng: orders
order_id                               string[python]
order_date                             datetime64[ns]
updated_at           datetime64[ns, Asia/Ho_Chi_Minh]
channel                                        object
shop_id                                string[python]
warehouse_id                           string[python]
status                                         object
video_id                               string[python]
platform_fee_vnd                                Int64
shipping_cost_vnd                               Int64
dtype: object

Bảng: order_items
order_item_id     string[python]
order_id          string[python]
sku               string[python]
qty                        Int64
unit_price_vnd             Int64
discount_vnd               Int64
refund_vnd                 Int64
unit_cost_vnd              Int64
dtype: object

Bảng: payments
transaction_id         string[python]
order_id               string[python]
settlement_date        datetime64[ns]
net_s

#### kiểm tra mã đơn giữa các bảng
##### Mọi order_id trong dòng hàng, thanh toán và tác vụ kho phải tồn tại trong orders

In [53]:
valid_order_ids = set(
    clean_api_tables["orders"]["order_id"].dropna()
)

for name in ["order_items", "payments", "warehouse_tasks"]:
    df = clean_api_tables[name]

    invalid_orders = df.loc[
        ~df["order_id"].isin(valid_order_ids)
    ]

    print(f"{name}: {len(invalid_orders)} dòng không khớp mã đơn")

    if not invalid_orders.empty:
        display(invalid_orders)

order_items: 0 dòng không khớp mã đơn
payments: 0 dòng không khớp mã đơn
warehouse_tasks: 0 dòng không khớp mã đơn


#### kiểm tra SKU và video với danh mục Sheets
##### Đơn hàng không có video_id được giữ nguyên và không tính là lỗi. Đơn đã gán video nhưng mã không có trong content mới cần kiểm tra.

In [54]:
valid_skus = set(clean_sheet_tables["products"]["sku"].dropna())
valid_videos = set(clean_sheet_tables["content"]["video_id"].dropna())

# 1. SKU trên dòng hàng phải có trong products
items = clean_api_tables["order_items"]
invalid_skus = items.loc[~items["sku"].isin(valid_skus)]

# 2. Video chạy Ads phải có trong content
ads = clean_api_tables["ads_daily"]
invalid_ads_videos = ads.loc[~ads["video_id"].isin(valid_videos)]

# 3. Đơn có gán video thì video phải có trong content
orders = clean_api_tables["orders"]
has_video = orders["video_id"].fillna("").str.strip().ne("")

invalid_order_videos = orders.loc[
    has_video & ~orders["video_id"].isin(valid_videos)
]

checks = {
    "order_items — SKU không khớp": invalid_skus,
    "ads_daily — video không khớp": invalid_ads_videos,
    "orders — video không khớp": invalid_order_videos
}

for label, errors in checks.items():
    print(f"{label}: {len(errors)} dòng")

    if not errors.empty:
        display(errors)

order_items — SKU không khớp: 0 dòng
ads_daily — video không khớp: 0 dòng
orders — video không khớp: 0 dòng


<div style="border:3px solid black;
            border-radius:10px;
            background-color:#FBE5D9;
            padding:20px 28px;
            font-size:110%;
            line-height:1.8;
            color:black;">
    
<p><b>Kiểm tra trạng thái đơn và số lượng bán</b></p>
<ul>

Quy tắc:
- Trạng thái đơn chỉ gồm completed, returned, cancelled.
- qty phải là số nguyên lớn hơn 0.
- Giá bán, giá vốn phải lớn hơn 0.
- Giảm giá và hoàn tiền không được âm.

</div>

In [56]:
orders = clean_api_tables["orders"]
items = clean_api_tables["order_items"]

invalid_status = orders.loc[
    ~orders["status"].isin(["completed", "returned", "cancelled"])
]

numeric_cols = [
    "qty", "unit_price_vnd", "unit_cost_vnd",
    "discount_vnd", "refund_vnd"
]

invalid_items = items.loc[
    items[numeric_cols].isna().any(axis=1)
    | (items["qty"] <= 0)
    | (items["qty"] % 1 != 0)
    | (items["unit_price_vnd"] <= 0)
    | (items["unit_cost_vnd"] <= 0)
    | (items["discount_vnd"] < 0)
    | (items["refund_vnd"] < 0)
    | (
        items["discount_vnd"] + items["refund_vnd"]
        > items["qty"] * items["unit_price_vnd"]
    )
]

print("Đơn có trạng thái không hợp lệ:", len(invalid_status))
print("Dòng hàng không hợp lệ:", len(invalid_items))

if not invalid_status.empty:
    display(invalid_status)

if not invalid_items.empty:
    display(invalid_items)

Đơn có trạng thái không hợp lệ: 0
Dòng hàng không hợp lệ: 0


#### Kiểm tra ads
##### Quy tắc kiểm tra:
- Các cột số không thiếu dữ liệu.
- Các giá trị không âm.
- clicks không vượt impressions.

In [57]:
ads = clean_api_tables["ads_daily"]
ads_cols = api_numeric_columns["ads_daily"]

invalid_ads = ads.loc[
    ads[ads_cols].isna().any(axis=1)
    | ads[ads_cols].lt(0).any(axis=1)
    | (ads["clicks"] > ads["impressions"])
]

print("Số dòng Ads không hợp lệ:", len(invalid_ads))
display(invalid_ads)

Số dòng Ads không hợp lệ: 0


,ad_record_id,date,advertiser_id,ad_id,video_id,impressions,clicks,spend_vnd,platform_attributed_orders,platform_attributed_revenue_vnd,attribution_window


#### Kiểm tra riêng bảng tác vụ kho
##### Quy tắc kiểm tra:
- Các cột số không thiếu dữ liệu.
- Loại tác vụ là PACK.
- units là số nguyên lớn hơn 0.
- duration_minutes lớn hơn 0.
- error_flag chỉ nhận 0 hoặc 1.

In [58]:
tasks = clean_api_tables["warehouse_tasks"]
task_cols = ["units", "duration_minutes", "error_flag"]

invalid_tasks = tasks.loc[
    tasks[task_cols].isna().any(axis=1)
    | ~tasks["task_type"].eq("PACK")
    | (tasks["units"] <= 0)
    | (tasks["units"] % 1 != 0)
    | (tasks["duration_minutes"] <= 0)
    | ~tasks["error_flag"].isin([0, 1])
]

print("Số tác vụ kho không hợp lệ:", len(invalid_tasks))
display(invalid_tasks)

Số tác vụ kho không hợp lệ: 0


,task_id,order_id,date,employee_id,warehouse_id,task_type,units,duration_minutes,error_flag


#### Kiểm tra ngày đặt đơn trong bảng orders

In [59]:
orders = clean_api_tables["orders"]

invalid_order_dates = orders.loc[
    orders["order_date"].isna()
    | (orders["order_date"] < pd.Timestamp("2026-01-01"))
    | (orders["order_date"] >= pd.Timestamp("2026-04-01"))
]

print("Số đơn có ngày không hợp lệ:", len(invalid_order_dates))

display(
    invalid_order_dates[
        ["order_id", "order_date", "status"]
    ]
)

Số đơn có ngày không hợp lệ: 0


,order_id,order_date,status


#### Kiểm tra bảng payments
##### Quy tắc: tiền thanh toán dự kiến = doanh thu ròng − phí sàn − vận chuyển.

In [60]:
payments = clean_api_tables["payments"]

payout_cols = [
    "net_sales_vnd",
    "platform_fee_vnd",
    "shipping_cost_vnd",
    "expected_payout_vnd",
    "actual_payout_vnd"
]

calculated_payout = (
    payments["net_sales_vnd"]
    - payments["platform_fee_vnd"]
    - payments["shipping_cost_vnd"]
)

invalid_payments = payments.loc[
    payments[payout_cols].isna().any(axis=1)
    | payments["expected_payout_vnd"].ne(calculated_payout)
]

print("Số dòng sai công thức thanh toán:", len(invalid_payments))
display(invalid_payments)

Số dòng sai công thức thanh toán: 0


,transaction_id,order_id,settlement_date,net_sales_vnd,platform_fee_vnd,shipping_cost_vnd,expected_payout_vnd,actual_payout_vnd


#### Xuất data (các bảng API đã clean)

In [61]:
clean_dir = ROOT / "output" / "clean"
clean_dir.mkdir(parents=True, exist_ok=True)

for name in [
    "orders",
    "order_items",
    "payments",
    "warehouse_tasks",
    "ads_daily"
]:
    # Tạo bản xuất để không đổi kiểu dữ liệu trong bộ nhớ
    export_df = clean_api_tables[name].copy()

    for col in api_date_columns[name]:
        export_df[col] = export_df[col].dt.strftime("%Y-%m-%d")

    if name == "orders":
        export_df["updated_at"] = export_df["updated_at"].map(
            lambda value: value.isoformat() if pd.notna(value) else ""
        )

    export_df.to_csv(
        clean_dir / f"{name}.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Đã lưu {name}.csv: {len(export_df):,} dòng")

# Lưu log chuẩn hóa mã và loại bản sao
reports_dir = ROOT / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame(api_cleaning_log).to_csv(
    reports_dir / "cleaning_log_api.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nĐã lưu dữ liệu và log làm sạch API.")

Đã lưu orders.csv: 6,000 dòng
Đã lưu order_items.csv: 12,105 dòng
Đã lưu payments.csv: 6,000 dòng
Đã lưu warehouse_tasks.csv: 5,530 dòng
Đã lưu ads_daily.csv: 6,000 dòng

Đã lưu dữ liệu và log làm sạch API.


# Tạo database company.db

#### Kiểm tra lại các file đã clean

In [62]:
expected_rows = {
    "receipts": 1200,
    "attendance": 3600,
    "inventory_movements": 15788,
    "products": 1200,
    "content": 1200,
    "stock_counts": 2400,
    "orders": 6000,
    "order_items": 12105,
    "payments": 6000,
    "warehouse_tasks": 5530,
    "ads_daily": 6000
}

clean_dir = ROOT / "output" / "clean"

for name, expected in expected_rows.items():
    path = clean_dir / f"{name}.csv"
    assert path.is_file(), f"Thiếu file: {name}.csv"

    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    assert len(df) == expected, (
        f"{name}: có {len(df)} dòng, cần {expected}"
    )

    print(f"✓ {name}: {len(df):,} dòng")

print("\nĐã kiểm tra đủ 11 file, số dòng đều khớp.")

✓ receipts: 1,200 dòng
✓ attendance: 3,600 dòng
✓ inventory_movements: 15,788 dòng
✓ products: 1,200 dòng
✓ content: 1,200 dòng
✓ stock_counts: 2,400 dòng
✓ orders: 6,000 dòng
✓ order_items: 12,105 dòng
✓ payments: 6,000 dòng
✓ warehouse_tasks: 5,530 dòng
✓ ads_daily: 6,000 dòng

Đã kiểm tra đủ 11 file, số dòng đều khớp.


In [63]:
import sqlite3
import pandas as pd

db_path = ROOT / "output" / "company.db"
conn = sqlite3.connect(db_path)

try:
    for name in expected_rows:
        csv_path = ROOT / "output" / "clean" / f"{name}.csv"

        # Giữ các cột mã dạng chuỗi
        columns = pd.read_csv(csv_path, nrows=0).columns
        id_types = {
            col: str
            for col in columns
            if col.endswith("_id") or col == "sku"
        }

        df = pd.read_csv(
            csv_path,
            dtype=id_types,
            keep_default_na=False
        )

        df.to_sql(
            name,
            conn,
            if_exists="replace",
            index=False
        )

        print(f"Đã nạp {name}: {len(df):,} dòng")

    conn.commit()

    display(pd.read_sql_query(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name;
        """,
        conn
    ))

finally:
    conn.close()

print("\nDatabase:", db_path.resolve())

Đã nạp receipts: 1,200 dòng
Đã nạp attendance: 3,600 dòng
Đã nạp inventory_movements: 15,788 dòng
Đã nạp products: 1,200 dòng
Đã nạp content: 1,200 dòng
Đã nạp stock_counts: 2,400 dòng
Đã nạp orders: 6,000 dòng
Đã nạp order_items: 12,105 dòng
Đã nạp payments: 6,000 dòng
Đã nạp warehouse_tasks: 5,530 dòng
Đã nạp ads_daily: 6,000 dòng


,name
0,ads_daily
1,attendance
2,content
3,inventory_movements
4,order_items
5,orders
6,payments
7,products
8,receipts
9,stock_counts



Database: /Users/hieutran/Downloads/DA_A_Z/output/company.db
